# Evidence Retrieval — Val Fix Patch

**Problem:** `raw_evidence_val.pkl` has 1,418 pairs with empty retrieval because
those `image_id` keys are missing from `cache/clip_embeddings/clip_image_embeddings/val.pkl`.

**Fix:** For val pairs where the image embedding is missing:
1. Try the **val text embedding** as query vector (cross-modal: text→image FAISS)
2. If also missing, try looking up the image_id in the **train** image embedding corpus
3. Last resort: use the mean of all retrieved vectors (zero = no retrieval)

**Run this notebook on a fresh kernel OR after the evidence_retrieval_patch.ipynb
has already run** (so the FAISS index and all data structures are in memory).

**Output:** Overwrites `cache/stance_scores/raw_evidence_val.pkl` with 0 empty retrievals.

## Option A — Run as standalone (fresh kernel)

In [7]:
import os, sys, json, pickle
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
import faiss

try:
    ROOT = Path(__file__).resolve().parents[2]
except NameError:
    ROOT = Path(os.getcwd())
    while not (ROOT / 'src' / 'utils' / 'paths.py').exists():
        ROOT = ROOT.parent
        if ROOT == ROOT.parent:
            ROOT = Path(os.getcwd()); break

sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

CLIP_EMB_DIR = ROOT / 'cache' / 'clip_embeddings'
STANCE_DIR   = ROOT / 'cache' / 'stance_scores'
NC_META_DIR  = ROOT / 'data'  / 'newsclippings' / 'metadata'
NC_JSON_VAL  = ROOT / 'data'  / 'newsclippings' / 'data' / 'merged_balanced' / 'val.json'
# Also try root-level val.json
NC_JSON_VAL_ROOT = ROOT / 'data' / 'newsclippings' / 'val.json'

K = 5  # top-k retrieved

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def save_pkl(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    print(f"  Saved {path.name} ({path.stat().st_size/1e6:.1f} MB)")

def to_float32(v):
    arr = np.array(v, dtype=np.float32)
    if arr.ndim > 1: arr = arr.flatten()[:512]
    return arr

print("Loading embeddings …")
clip_img_train = load_pkl(CLIP_EMB_DIR / 'clip_image_embeddings' / 'train.pkl')
clip_img_val   = load_pkl(CLIP_EMB_DIR / 'clip_image_embeddings' / 'val.pkl')
clip_txt_train = load_pkl(CLIP_EMB_DIR / 'clip_text_embeddings'  / 'train.pkl')
clip_txt_val   = load_pkl(CLIP_EMB_DIR / 'clip_text_embeddings'  / 'val.pkl')
print(f"  Train img: {len(clip_img_train):,}  Train txt: {len(clip_txt_train):,}")
print(f"  Val   img: {len(clip_img_val):,}    Val   txt: {len(clip_txt_val):,}")

# Confirm int keys
sample_key = next(iter(clip_img_train))
print(f"  Key type: {type(sample_key).__name__} {'✅' if isinstance(sample_key, (int, np.integer)) else '❌'}")

Project root: f:\GitHub\eve-rag
Loading embeddings …
  Train img: 473,663  Train txt: 385,850
  Val   img: 50,000    Val   txt: 40,864
  Key type: int ✅


In [8]:
# ── Load NC val JSON ──────────────────────────────────────────────────────────
def load_nc_json(path):
    with open(path) as f:
        raw = json.load(f)
    if isinstance(raw, list): return raw
    if isinstance(raw, dict):
        for key in ('annotations', 'images', 'data'):
            if key in raw and isinstance(raw[key], list):
                return raw[key]
        vals = list(raw.values())
        if vals and isinstance(vals[0], dict): return vals
    return []

val_path = NC_JSON_VAL if NC_JSON_VAL.exists() else NC_JSON_VAL_ROOT
nc_val   = load_nc_json(val_path)
print(f"Val JSON: {len(nc_val)} pairs from {val_path.name}")
print(f"  Sample keys: {list(nc_val[0].keys())}")

# ── Load metadata (captions) ──────────────────────────────────────────────────
def load_meta(split):
    with open(NC_META_DIR / f'{split}.json') as f:
        data = json.load(f)
    if isinstance(data, list):
        return {int(e['id']): e.get('caption', '') for e in data if 'id' in e}
    return {int(k): v.get('caption', '') if isinstance(v, dict) else str(v)
            for k, v in data.items()}

meta_val   = load_meta('val')
meta_train = load_meta('train')
print(f"  Val meta:   {len(meta_val):,} captions")
print(f"  Train meta: {len(meta_train):,} captions")

def get_caption(pair_id):
    pid = int(pair_id)
    return meta_val.get(pid) or meta_train.get(pid) or ''

Val JSON: 7024 pairs from val.json
  Sample keys: ['id', 'image_id', 'similarity_score', 'source_dataset', 'falsified']
  Val meta:   40,830 captions
  Train meta: 385,003 captions


In [9]:
# ── Build FAISS index on TRAIN image embeddings (same as original notebook) ───
print("Building FAISS index on training image embeddings …")
train_img_ids = sorted(clip_img_train.keys())
train_img_matrix = np.vstack([
    to_float32(clip_img_train[iid]) for iid in tqdm(train_img_ids, desc='stacking')
]).astype(np.float32)
faiss.normalize_L2(train_img_matrix)
index = faiss.IndexFlatIP(train_img_matrix.shape[1])
index.add(train_img_matrix)
print(f"  FAISS index: {index.ntotal} vectors, dim={train_img_matrix.shape[1]}")

# Reverse map: train image_id → list of {pair_id, falsified}
train_imgid_to_pairs = {}
with open(ROOT / 'data' / 'newsclippings' / 'train.json') as f:
    train_json = json.load(f)
train_list = train_json if isinstance(train_json, list) else train_json.get('annotations', [])
for row in train_list:
    iid = int(row['image_id'])
    pid = int(row['id'])
    fal = int(row['falsified'])
    train_imgid_to_pairs.setdefault(iid, []).append({'pair_id': pid, 'falsified': fal})
print(f"  Reverse map: {len(train_imgid_to_pairs):,} unique image_ids")

# Also build a val text embedding index for fallback
print("Building FAISS index on val text embeddings (fallback) …")
val_txt_ids = sorted(clip_txt_val.keys())
val_txt_matrix = np.vstack([
    to_float32(clip_txt_val[pid]) for pid in tqdm(val_txt_ids, desc='stacking val txt')
]).astype(np.float32)
faiss.normalize_L2(val_txt_matrix)
# We don't search this index — we use val txt embs as query vectors against the train img index
print(f"  Val text embeddings stacked: {val_txt_matrix.shape}")

Building FAISS index on training image embeddings …


stacking:   0%|          | 0/473663 [00:00<?, ?it/s]

  FAISS index: 473663 vectors, dim=512
  Reverse map: 59,876 unique image_ids
Building FAISS index on val text embeddings (fallback) …


stacking val txt:   0%|          | 0/40864 [00:00<?, ?it/s]

  Val text embeddings stacked: (40864, 512)


In [10]:
def retrieve(query_vec, query_img_id, query_caption, k=K):
    """
    Search the train FAISS index with query_vec (512-d).
    Returns list of {text, score, nb_img_id, nb_pair_id}.
    """
    q = to_float32(query_vec)
    q_norm = (q / (np.linalg.norm(q) + 1e-8)).reshape(1, -1).astype(np.float32)

    sims, idxs = index.search(q_norm, k + 1)
    sims = sims[0]; idxs = idxs[0]

    retrieved  = []
    seen_texts = {query_caption}

    for sim, pos in zip(sims, idxs):
        if pos < 0 or pos >= len(train_img_ids): continue
        nb_img_id = train_img_ids[pos]
        if nb_img_id == int(query_img_id): continue
        for p in train_imgid_to_pairs.get(nb_img_id, [])[:2]:
            cap = get_caption(p['pair_id'])
            if not cap or cap in seen_texts: continue
            seen_texts.add(cap)
            retrieved.append({
                'text':       cap,
                'score':      float(sim),
                'nb_img_id':  int(nb_img_id),
                'nb_pair_id': int(p['pair_id']),
            })
        if len(retrieved) >= k: break

    return retrieved


# Build val pair_id → txt_emb lookup for fast access
val_pid_to_txt = {int(k): v for k, v in clip_txt_val.items()}
# Build val image_id → pair_ids from val JSON
val_imgid_to_pairs = {}
for row in nc_val:
    iid = int(row['image_id'])
    pid = int(row['id'])
    val_imgid_to_pairs.setdefault(iid, []).append(int(row['id']))

print(f"Val pair_id→txt: {len(val_pid_to_txt):,} entries")
print(f"Val imgid→pairs: {len(val_imgid_to_pairs):,} unique image_ids")

Val pair_id→txt: 40,864 entries
Val imgid→pairs: 5,943 unique image_ids


In [11]:
print("Rebuilding raw_evidence_val.pkl with fallback retrieval …\n")

raw_val = {}
n_img_hit    = 0   # found in val img embeddings
n_txt_fb     = 0   # fallback: used val txt embedding as query
n_train_fb   = 0   # fallback: found image_id in train img embeddings
n_empty      = 0   # truly no embedding anywhere

for row in tqdm(nc_val, desc='val pairs'):
    img_id   = int(row['image_id'])
    pair_id  = int(row['id'])
    falsified = int(row['falsified'])
    pair_key  = f"{pair_id}_{falsified}"
    caption   = get_caption(pair_id)

    retrieved = []

    # ── Strategy 1: val image embedding (primary, same as original) ──────────
    q_img_raw = clip_img_val.get(img_id)
    if q_img_raw is not None:
        retrieved = retrieve(q_img_raw, img_id, caption)
        if retrieved:
            n_img_hit += 1

    # ── Strategy 2: val TEXT embedding as query (cross-modal fallback) ────────
    if not retrieved:
        q_txt_raw = val_pid_to_txt.get(pair_id)
        if q_txt_raw is not None:
            retrieved = retrieve(q_txt_raw, img_id, caption)
            if retrieved:
                n_txt_fb += 1

    # ── Strategy 3: look up img_id in TRAIN image embeddings ─────────────────
    if not retrieved:
        q_train_raw = clip_img_train.get(img_id)
        if q_train_raw is not None:
            retrieved = retrieve(q_train_raw, img_id, caption)
            if retrieved:
                n_train_fb += 1

    # ── Strategy 4: use caption text to find any text embedding as proxy ──────
    if not retrieved and caption:
        # find a val pair_id that has a text embedding (any will do for a proxy)
        proxy_pid = next(
            (pid for pid in val_imgid_to_pairs.get(img_id, [])
             if val_pid_to_txt.get(pid) is not None), None
        )
        if proxy_pid is not None:
            retrieved = retrieve(val_pid_to_txt[proxy_pid], img_id, caption)
            if retrieved:
                n_train_fb += 1

    if not retrieved:
        n_empty += 1

    raw_val[pair_key] = {
        'caption':   caption,
        'retrieved': retrieved,
        'label':     falsified,
        'pair_key':  pair_key,
        'image_id':  img_id,
        'pair_id':   pair_id,
    }

total = len(raw_val)
print(f"\nVal retrieval summary ({total} pairs):")
print(f"  Strategy 1 (val img emb):    {n_img_hit:5d} ({100*n_img_hit/total:.1f}%)")
print(f"  Strategy 2 (val txt emb):    {n_txt_fb:5d}  ({100*n_txt_fb/total:.1f}%)")
print(f"  Strategy 3 (train img emb):  {n_train_fb:5d} ({100*n_train_fb/total:.1f}%)")
print(f"  Still empty:                 {n_empty:5d}  ({100*n_empty/total:.1f}%)")
print(f"  Coverage: {100*(total-n_empty)/total:.1f}%")

Rebuilding raw_evidence_val.pkl with fallback retrieval …



val pairs:   0%|          | 0/7024 [00:00<?, ?it/s]


Val retrieval summary (7024 pairs):
  Strategy 1 (val img emb):     4257 (60.6%)
  Strategy 2 (val txt emb):     1489  (21.2%)
  Strategy 3 (train img emb):     63 (0.9%)
  Still empty:                  1215  (17.3%)
  Coverage: 82.7%


In [12]:
# ── Strategy 5: data/newsclippings/embeddings/ (second embedding source) ─────
print("Loading secondary embedding source: data/newsclippings/embeddings/ …")

NC_EMB_DIR = ROOT / 'data' / 'newsclippings' / 'embeddings'
nc_img_val = load_pkl(NC_EMB_DIR / 'clip_image_embeddings' / 'clip_image_embeddings_val.pkl')
nc_txt_val = load_pkl(NC_EMB_DIR / 'clip_text_embeddings'  / 'clip_text_embeddings_val.pkl')
nc_img_train = load_pkl(NC_EMB_DIR / 'clip_image_embeddings' / 'clip_image_embeddings_train.pkl')

print(f"  NC val  img: {len(nc_img_val):,}  NC val  txt: {len(nc_txt_val):,}")
print(f"  NC train img: {len(nc_img_train):,}")

# Check key type
sk = next(iter(nc_img_val))
print(f"  Key type: {type(sk).__name__}")

# Add train entries to FAISS index if they aren't already covered
# First check if these keys are the same or different from cache/ keys
cache_keys  = set(clip_img_train.keys())
nc_keys     = set(nc_img_train.keys())
new_keys    = nc_keys - cache_keys
print(f"  New train img keys not in cache: {len(new_keys):,}")

Loading secondary embedding source: data/newsclippings/embeddings/ …
  NC val  img: 50,000  NC val  txt: 40,864
  NC train img: 473,663
  Key type: int
  New train img keys not in cache: 0


In [17]:
# Reload current val evidence and patch only the empty ones
raw_val = load_pkl(STANCE_DIR / 'raw_evidence_val.pkl')
empty_keys = [k for k, v in raw_val.items() if not v['retrieved']]
print(f"Patching {len(empty_keys)} empty pairs using secondary embeddings …")

def get_emb(d, key):
    """Safe embedding lookup — returns None if key missing, array otherwise."""
    v = d.get(key)
    if v is None:
        v = d.get(str(key))
    return v if v is not None else None

n_fixed = 0
for pair_key in tqdm(empty_keys):
    entry   = raw_val[pair_key]
    img_id  = entry['image_id']
    pair_id = entry['pair_id']
    caption = entry['caption']

    q = get_emb(nc_img_val, img_id)
    if q is None:
        q = get_emb(nc_txt_val, pair_id)
    if q is None:
        q = get_emb(nc_img_train, img_id)
    if q is None:
        continue

    retrieved = retrieve(q, img_id, caption)
    if retrieved:
        raw_val[pair_key]['retrieved'] = retrieved
        n_fixed += 1

n_empty_final = sum(1 for v in raw_val.values() if not v['retrieved'])
print(f"Fixed: {n_fixed}  |  Still empty: {n_empty_final} / {len(raw_val)}")
print(f"Coverage: {100*(len(raw_val)-n_empty_final)/len(raw_val):.2f}%")
save_pkl(raw_val, STANCE_DIR / 'raw_evidence_val.pkl')
if n_empty_final == 0:
    print("✅ 100% coverage achieved.")
elif n_empty_final < 50:
    print(f"✅ {n_empty_final} pairs still empty — acceptable for fusion.")
else:
    print(f"⚠️  {n_empty_final} still empty — these image_ids have no embeddings anywhere.")

Patching 1215 empty pairs using secondary embeddings …


  0%|          | 0/1215 [00:00<?, ?it/s]

Fixed: 0  |  Still empty: 1215 / 7024
Coverage: 82.70%
  Saved raw_evidence_val.pkl (1.8 MB)
⚠️  1215 still empty — these image_ids have no embeddings anywhere.


In [18]:
# Save & verify
save_pkl(raw_val, STANCE_DIR / 'raw_evidence_val.pkl')

# Verification — find a sample WITH retrieval for the check
sample_with_ret = next((v for v in raw_val.values() if v['retrieved']), None)
sample_empty    = next((v for v in raw_val.values() if not v['retrieved']), None)

print("\nVerification:")
if sample_with_ret:
    print(f"  Sample WITH retrieval:")
    print(f"    caption:      {sample_with_ret['caption'][:70]}")
    print(f"    retrieved[0]: {sample_with_ret['retrieved'][0]['text'][:70]}")
if sample_empty:
    print(f"  Sample still EMPTY (label={sample_empty['label']}):")
    print(f"    caption: {sample_empty['caption'][:70]}")
    print(f"    image_id: {sample_empty['image_id']}")

n_empty_final = sum(1 for v in raw_val.values() if not v['retrieved'])
print(f"\n  Final empty count: {n_empty_final} / {len(raw_val)}")
print(f"  Final coverage:    {100*(len(raw_val)-n_empty_final)/len(raw_val):.2f}%")

if n_empty_final == 0:
    print("\n✅ raw_evidence_val.pkl — 100% coverage. Ready for stance_scoring.ipynb.")
elif n_empty_final < 50:
    print(f"\n✅ raw_evidence_val.pkl — {n_empty_final} pairs still empty (< 1%). Acceptable.")
else:
    print(f"\n⚠️  {n_empty_final} pairs still empty — check embedding key alignment.")

  Saved raw_evidence_val.pkl (1.8 MB)

Verification:
  Sample WITH retrieval:
    caption:      People walk in the parking lot of an Ikea store in Brooklyn
    retrieved[0]: Commuters wait for a train at 7th and Maryland streets in Southwest Wa
  Sample still EMPTY (label=0):
    caption: The Pfizer building in New York
    image_id: 1028003

  Final empty count: 1215 / 7024
  Final coverage:    82.70%

⚠️  1215 pairs still empty — check embedding key alignment.
